# 01_load_communities.py

```text
WHAT : Turns the BushTel community-profile JSON into one tidy table, one row per community.
READS: data/raw/bushtel/Com_BushTel_Profile_CMC_2024.json
WRITES: data/processed/01_communities.csv        (plain table, one row per community)
        data/processed/01_communities.geojson    (same rows, as mappable points, EPSG:4326)
        data/processed/01_communities.columns.md  (one line describing each column)
WHY  : BushTel (2024) is the best available list of *where* the ~792 NT remote communities
       are and what type they are. Every later step starts from this table. We only clean
       and rename here - no analysis, no joins.
RUN  : python src/01_load_communities.py        (about 1 second)
```

In [75]:
import importlib
import json
import geopandas as gpd
import pandas as pd

### Read the raw JSON.

   BushTel is a GeoJSON FeatureCollection: each "feature" is one community, with a
   Point geometry (lon, lat) and a "properties" dict holding the attributes.

In [76]:
bushtel_file = 'data/raw/bushtel/Com_BushTel_Profile_CMC_2024.json'
with open(bushtel_file, encoding="utf-8") as fh:
    raw = json.load(fh)

features = raw["features"]
print(f"read {len(features)} community records from BushTel")

read 792 community records from BushTel


We keep BushTel's own identifiers and descriptive columns unchanged in meaning,
   just given clearer names. The population field is renamed to make clear it is
   BushTel's own 2024 figure (a later script adds a Census-based estimate alongside it).

In [77]:
rows = []
for feature in features:
    prop = feature["properties"]
    lon, lat = feature["geometry"]["coordinates"][:2]  # BushTel stores [longitude, latitude]

    rows.append({
        "community_id": prop["community_id"],                 # BushTel's stable numeric id
        "community_name": prop["community_name"],
        "community_aliases": prop.get("community_aliases") or "",  # comma-separated, often blank
        "community_type": prop["community_type"],             # e.g. "Family Outstation", "Town"
        "latitude": lat,
        "longitude": lon,
        "land_council": prop.get("land_council"),
        "local_govt_council": prop.get("local_govt_council"),
        "ntg_region": prop.get("ntg_region"),
        "ward": prop.get("ward"),
        "electorate": prop.get("electorate"),
        "main_language": prop.get("main_language"),
        # BushTel's own population count and where BushTel says that number came from.
        "population_bushtel_2024": prop.get("population_count"),
        "population_bushtel_source": prop.get("population_source"),
        "bushtel_url": prop.get("bushtel_url"),
    })

communities = pd.DataFrame(rows)

In [78]:
communities.head()

,community_id,community_name,community_aliases,community_type,latitude,longitude,land_council,local_govt_council,ntg_region,ward,electorate,main_language,population_bushtel_2024,population_bushtel_source,bushtel_url
0,1022,10 MILE,"PALPANG,TEN MILE",Family Outstation,-20.40628,134.75934,CENTRAL LAND COUNCIL,BARKLY,BARKLY,ALYAWARR,BARKLY,Not recorded,8.0,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/1022
1,1004,10 MILE OUTSTATION,TEN MILE OUTSTATION,Family Outstation,-22.46667,132.37658,CENTRAL LAND COUNCIL,CENTRAL DESERT,CENTRAL AUSTRALIA,ANMATJERE,GWOJA,Not recorded,10.0,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/1004
2,783,16 MILE CAMP,"ATHENGE-LHERE,BOND SPRINGS",Family Outstation,-23.45764,133.83248,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,RODINGA,NAMATJIRA,Not recorded,NaN,Not recorded,https://bushtel.nt.gov.au/profile/783
3,24993,20 MILE,WUMAYALINJA,Family Outstation,-16.08535,136.54826,NORTHERN LAND COUNCIL,ROPER GULF,BIG RIVERS,SOUTH WEST GULF,BARKLY,Not recorded,6.0,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/24993
4,1006,4 MILE CAMP,"FOUR MILE CAMP,NGUIU 4 MILE CAMP",Family Outstation,-11.77798,130.59548,TIWI LAND COUNCIL,TIWI ISLANDS,TOP END,BATHURST ISLAND WARD,ARAFURA,Not recorded,7.0,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/1006


In [79]:
communities.info()

<class 'pandas.DataFrame'>
RangeIndex: 792 entries, 0 to 791
Data columns (total 15 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   community_id               792 non-null    int64  
 1   community_name             792 non-null    str    
 2   community_aliases          792 non-null    str    
 3   community_type             792 non-null    str    
 4   latitude                   792 non-null    float64
 5   longitude                  792 non-null    float64
 6   land_council               792 non-null    str    
 7   local_govt_council         791 non-null    str    
 8   ntg_region                 792 non-null    str    
 9   ward                       792 non-null    str    
 10  electorate                 792 non-null    str    
 11  main_language              792 non-null    str    
 12  population_bushtel_2024    457 non-null    float64
 13  population_bushtel_source  792 non-null    str    
 14  busht

### Tidy the column types.

   population_count comes through as a number or as null; make it a nullable integer
   so "no figure" stays distinct from "a figure of 0".

In [80]:
communities["population_bushtel_2024"] = (
    pd.to_numeric(communities["population_bushtel_2024"], errors="coerce").astype("Int64")
)
communities["community_id"] = communities["community_id"].astype(int)

# Sort by id so the output file has a stable, predictable order.
communities = communities.sort_values("community_id").reset_index(drop=True)

In [81]:
communities['community_id'].nunique()

792

In [82]:
assert communities["community_id"].is_unique

In [83]:
n_total = len(communities)
n_with_pop = int(communities["population_bushtel_2024"].notna().sum())
n_without_pop = n_total - n_with_pop

print()
print(f"{n_total} communities")
print(f"  {n_with_pop} have a BushTel population figure, {n_without_pop} do not")
print("  community types:")
for ctype, count in communities["community_type"].value_counts().items():
    print(f"    {count:4d}  {ctype}")
print("  population source (BushTel's own note):")
for src, count in communities["population_bushtel_source"].value_counts(dropna=False).items():
    print(f"    {count:4d}  {src}")


792 communities
  457 have a BushTel population figure, 335 do not
  community types:
     630  Family Outstation
      59  Major
      45  Town Camp
      37  Minor
      13  Village
       7  Town
       1  City
  population source (BushTel's own note):
     379  Homelands Service Provider Report 2023
     335  Not recorded
      78  Based on ABS 2021 Census SA1


# 02_attach_census.py

```text
WHAT : Builds the two tables the rest of the project reports on.
       (1) VILLAGE table  - the 792 BushTel communities, each tagged with the ABS SA1
           and ILOC area it falls inside. No census population is put on a village.
       (2) SA1 table      - every one of the 649 NT SA1s, with its 2021 Census
           population, Indigenous count/share, median household income, an overcrowding
           proxy, its area and remoteness class, and the list of villages inside it.
           This is the "reporting" table; gap measures (people per serving asset, etc.)
           are added later in 08 once the asset layers exist.
READS: data/processed/01_communities.csv
       data/external/abs_boundaries/SA1_2021_AUST_GDA2020_SHP/SA1_2021_AUST_GDA2020.shp
       data/external/abs_boundaries/ILOC_2021_AUST_GDA2020_SHP/ILOC_2021_AUST_GDA2020.shp
       data/external/abs_boundaries/RA_2021_AUST.xlsx           (SA1 -> remoteness + area)
       data/raw/census_gcp_2021/.../SA1/NT/2021Census_G01_NT_SA1.csv   (persons, Indigenous)
       data/raw/census_gcp_2021/.../SA1/NT/2021Census_G02_NT_SA1.csv   (medians, overcrowding)
WRITES: data/processed/02_villages_with_areas.csv        (+ .geojson, + .columns.md)
        data/processed/02_sa1_report.csv                 (+ .columns.md)
        data/processed/02_sa1_boundaries.geojson         (NT SA1 polygons, for 08 + the viewer)
WHY  : We changed the unit of analysis. Splitting an SA1's population across the outstations
       inside it was guesswork and made outstation-heavy areas look artificially crowded or
       empty. Instead:
         - the SA1 is the unit for POPULATION and equity numbers (matches how the ABS and
           the Digital Inclusion Index report, needs no splitting, and does not inflate a
           place just because it has many tiny outstations);
         - the VILLAGE is the unit for DISTANCE and the T1-T5 tier (one cell tower in a
           huge SA1 does not cover every outstation, so "is a service in reach" has to be
           asked per village).
RUN  : python src/02_attach_census.py        (about 15 seconds - the SA1 shapefile is large)

Census counts are place of usual residence on Census night, 9 August 2021, and small
counts are randomly adjusted by the ABS for confidentiality. Treat them as estimates.
```

### Write the two data outputs.

   - CSV: the table on its own, easiest to open and diff.
   - GeoJSON: the same rows as points in plain lat/long, for the map viewer and for
     the next script's spatial join.

In [84]:
# villages= communities["population_bushtel_2024"] = (
#     pd.to_numeric(communities["population_bushtel_2024"], errors="coerce").astype("Int64")
# )
villages = communities.copy()

In [85]:
village_points = gpd.GeoDataFrame(
    villages.copy(),
    geometry=gpd.points_from_xy(villages["longitude"], villages["latitude"]),
    crs='EPSG:4326',
)

Load the NT boundary polygons. The shapefiles are national; "where=" filters to NT as the file is read. Then reproject to plain lat/long to match the village points.

In [86]:
nt_only = "STE_NAME21 = 'Northern Territory'"

In [87]:
# loading census 2021 gcp SA1 data
cen21_sa1 = pd.read_csv('data/raw/census_gcp_2021/2021_GCP_all_for_NT_short-header/2021 Census GCP All Geographies for NT/SA1/NT/2021Census_G01_NT_SA1.csv', dtype={"SA1_CODE_2021": str})
cen21_sa1.head()

,SA1_CODE_2021,Tot_P_M,Tot_P_F,Tot_P_P,Age_0_4_yr_M,Age_0_4_yr_F,Age_0_4_yr_P,Age_5_14_yr_M,Age_5_14_yr_F,Age_5_14_yr_P,...,High_yr_schl_comp_Yr_8_belw_P,High_yr_schl_comp_D_n_g_sch_M,High_yr_schl_comp_D_n_g_sch_F,High_yr_schl_comp_D_n_g_sch_P,Count_psns_occ_priv_dwgs_M,Count_psns_occ_priv_dwgs_F,Count_psns_occ_priv_dwgs_P,Count_Persons_other_dwgs_M,Count_Persons_other_dwgs_F,Count_Persons_other_dwgs_P
0,70101100101,10,4,23,0,0,0,0,0,0,...,0,0,0,0,0,0,0,86,49,137
1,70101100206,7,0,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,5,0,5
2,70101100207,170,124,293,8,4,9,7,4,9,...,0,0,0,0,141,112,252,197,189,386
3,70101100208,188,165,351,3,4,9,9,19,31,...,4,0,0,0,169,152,322,24,22,49
4,70101100209,192,161,354,0,7,13,4,12,18,...,3,0,0,0,161,141,298,140,115,255


In [88]:
sa1_polys = gpd.read_file('data/external/abs_boundaries/SA1_2021_AUST_GDA2020_SHP/SA1_2021_AUST_GDA2020.shp', where=nt_only)
sa1_polys

,SA1_CODE21,CHG_FLAG21,CHG_LBL21,SA2_CODE21,SA2_NAME21,SA3_CODE21,SA3_NAME21,SA4_CODE21,SA4_NAME21,GCC_CODE21,GCC_NAME21,STE_CODE21,STE_NAME21,AUS_CODE21,AUS_NAME21,AREASQKM21,LOCI_URI21,geometry
0,70101100101,0,No change,701011001,Darwin Airport,70101,Darwin City,701,Darwin,7GDAR,Greater Darwin,7,Northern Territory,AUS,Australia,14.3244,http://linked.data.gov.au/dataset/asgsed3/SA1/...,"POLYGON ((130.87759 -12.42488, 130.87044 -12.4..."
1,70101100206,0,No change,701011002,Darwin City,70101,Darwin City,701,Darwin,7GDAR,Greater Darwin,7,Northern Territory,AUS,Australia,0.4568,http://linked.data.gov.au/dataset/asgsed3/SA1/...,"POLYGON ((130.8478 -12.45297, 130.84791 -12.45..."
2,70101100207,0,No change,701011002,Darwin City,70101,Darwin City,701,Darwin,7GDAR,Greater Darwin,7,Northern Territory,AUS,Australia,0.7414,http://linked.data.gov.au/dataset/asgsed3/SA1/...,"POLYGON ((130.84414 -12.46895, 130.84399 -12.4..."
3,70101100208,0,No change,701011002,Darwin City,70101,Darwin City,701,Darwin,7GDAR,Greater Darwin,7,Northern Territory,AUS,Australia,0.3234,http://linked.data.gov.au/dataset/asgsed3/SA1/...,"POLYGON ((130.84343 -12.46473, 130.84393 -12.4..."
4,70101100209,0,No change,701011002,Darwin City,70101,Darwin City,701,Darwin,7GDAR,Greater Darwin,7,Northern Territory,AUS,Australia,0.1139,http://linked.data.gov.au/dataset/asgsed3/SA1/...,"POLYGON ((130.84164 -12.46299, 130.84213 -12.4..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,70205106812,0,No change,702051068,Victoria River,70205,Katherine,702,Northern Territory - Outback,7RNTE,Rest of NT,7,Northern Territory,AUS,Australia,27280.9229,http://linked.data.gov.au/dataset/asgsed3/SA1/...,"POLYGON ((132.52289 -16.58328, 132.43109 -16.5..."
645,79797979991,0,No change,797979799,Migratory - Offshore - Shipping (NT),79797,Migratory - Offshore - Shipping (NT),797,Migratory - Offshore - Shipping (NT),79799,Migratory - Offshore - Shipping (NT),7,Northern Territory,AUS,Australia,NaN,http://linked.data.gov.au/dataset/asgsed3/SA1/...,None
646,79797979992,0,No change,797979799,Migratory - Offshore - Shipping (NT),79797,Migratory - Offshore - Shipping (NT),797,Migratory - Offshore - Shipping (NT),79799,Migratory - Offshore - Shipping (NT),7,Northern Territory,AUS,Australia,NaN,http://linked.data.gov.au/dataset/asgsed3/SA1/...,None
647,79797979993,0,No change,797979799,Migratory - Offshore - Shipping (NT),79797,Migratory - Offshore - Shipping (NT),797,Migratory - Offshore - Shipping (NT),79799,Migratory - Offshore - Shipping (NT),7,Northern Territory,AUS,Australia,NaN,http://linked.data.gov.au/dataset/asgsed3/SA1/...,None


In [89]:
# sa1_polys = gpd.read_file('data/external/abs_boundaries/SA1_2021_AUST_GDA2020_SHP/SA1_2021_AUST_GDA2020.shp', where=nt_only)
sa1_polys = sa1_polys[["SA1_CODE21", "SA2_NAME21", "SA3_NAME21", "SA4_NAME21", "geometry"]]
sa1_polys = sa1_polys.rename(columns={
    "SA1_CODE21": "sa1_code", "SA2_NAME21": "sa2_name",
    "SA3_NAME21": "sa3_name", "SA4_NAME21": "sa4_name",
}).to_crs('EPSG:4326')
print(f"{len(sa1_polys)} SA1 polygons in the NT")

649 SA1 polygons in the NT


In [90]:
sa1_polys.head()

,sa1_code,sa2_name,sa3_name,sa4_name,geometry
0,70101100101,Darwin Airport,Darwin City,Darwin,"POLYGON ((130.87759 -12.42488, 130.87044 -12.4..."
1,70101100206,Darwin City,Darwin City,Darwin,"POLYGON ((130.8478 -12.45297, 130.84791 -12.45..."
2,70101100207,Darwin City,Darwin City,Darwin,"POLYGON ((130.84414 -12.46895, 130.84399 -12.4..."
3,70101100208,Darwin City,Darwin City,Darwin,"POLYGON ((130.84343 -12.46473, 130.84393 -12.4..."
4,70101100209,Darwin City,Darwin City,Darwin,"POLYGON ((130.84164 -12.46299, 130.84213 -12.4..."


In [91]:
iloc_polys = gpd.read_file('data/external/abs_boundaries/ILOC_2021_AUST_GDA2020_SHP/ILOC_2021_AUST_GDA2020.shp', where=nt_only)
iloc_polys = iloc_polys[["ILO_CODE21", "ILO_NAME21", "geometry"]]
iloc_polys = iloc_polys.rename(
    columns={"ILO_CODE21": "iloc_code", "ILO_NAME21": "iloc_name"}
).to_crs('EPSG:4326')
print(f"{len(iloc_polys)} ILOC polygons in the NT")

189 ILOC polygons in the NT


In [92]:
iloc_polys.head()

,iloc_code,iloc_name,geometry
0,70300101,Alawa,"POLYGON ((130.87259 -12.38331, 130.8709 -12.38..."
1,70300102,Brinkin - Nakara,"POLYGON ((130.88407 -12.37141, 130.88421 -12.3..."
2,70300201,Anula,"POLYGON ((130.88847 -12.38499, 130.88923 -12.3..."
3,70300202,Wulagi,"POLYGON ((130.90074 -12.38241, 130.90162 -12.3..."
4,70300301,Bagot Community,"POLYGON ((130.8523 -12.41273, 130.85301 -12.41..."


Spatial join: which SA1 and which ILOC does each village point sit inside?

In [93]:
tagged = gpd.sjoin(
    village_points, sa1_polys[["sa1_code", "sa2_name", "geometry"]],
    how="left", predicate="within",
).drop(columns="index_right")

In [94]:
tagged = gpd.sjoin(
    tagged, iloc_polys, how="left", predicate="within",
).drop(columns="index_right")

In [95]:
tagged

,community_id,community_name,community_aliases,community_type,latitude,longitude,land_council,local_govt_council,ntg_region,ward,electorate,main_language,population_bushtel_2024,population_bushtel_source,bushtel_url,geometry,sa1_code,sa2_name,iloc_code,iloc_name
0,1,ADELAIDE BORE,"WOOLA,WOOLLA",Family Outstation,-22.21485,133.93607,CENTRAL LAND COUNCIL,CENTRAL DESERT,CENTRAL AUSTRALIA,ANMATJERE,BARKLY,Not recorded,14,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/1,POINT (133.93607 -22.21485),70201105410,Yuendumu - Anmatjere,70900301,Anmatjere - Surrounds
1,2,ALYUEN,"AILERON,ALUM,ALUM CAMP",Family Outstation,-22.66949,133.35147,CENTRAL LAND COUNCIL,CENTRAL DESERT,CENTRAL AUSTRALIA,ANMATJERE,BARKLY,Not recorded,12,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/2,POINT (133.35147 -22.66949),70201105410,Yuendumu - Anmatjere,70900301,Anmatjere - Surrounds
2,3,AKNGWERTNARRE,MORRIS SOAK,Town Camp,-23.69280,133.86120,CENTRAL LAND COUNCIL,ALICE SPRINGS,CENTRAL AUSTRALIA,Not recorded,BRAITLING,Not recorded,<NA>,Not recorded,https://bushtel.nt.gov.au/profile/3,POINT (133.8612 -23.6928),70201104513,Charles,70800206,Nyewente - Akngwertnarre - Namatjira
3,4,ALKIPI,"ALKIPIE,ALKPIE,MOUNT LARRY,YALKIPI",Family Outstation,-23.27356,131.82924,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,LURITJA PINTUBI,GWOJA,Not recorded,<NA>,Not recorded,https://bushtel.nt.gov.au/profile/4,POINT (131.82924 -23.27356),70201105310,Tanami,70901102,Papunya Outstations
4,5,RED SANDHILL,INTJIRRANGA,Family Outstation,-24.00974,132.85279,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,LJIRAPINTA,GWOJA,Not recorded,3,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/5,POINT (132.85279 -24.00974),70201105312,Tanami,70901403,Tjuwanpa Outstations
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,13339033,HONEYMOON BORE,No aliases recorded with NTG DIPL,Family Outstation,-21.66791,135.24665,CENTRAL LAND COUNCIL,BARKLY,BARKLY,ALYAWARR,BARKLY,Not recorded,2,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/13339033,POINT (135.24665 -21.66791),70201105209,Sandover - Plenty,70900201,Ampilatwatja and Outstations
788,14045564,BARNJARN,No aliases recorded with NTG DIPL,Family Outstation,-14.28264,132.08321,NORTHERN LAND COUNCIL,ROPER GULF,BIG RIVERS,NYIRRANGGULUNG,ARNHEM,Not recorded,8,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/14045564,POINT (132.08321 -14.28264),70205106503,Elsey,70500303,Elsey Roper - Surrounds
789,14153168,SALT BORE,No aliases recorded with NTG DIPL,Family Outstation,-23.77120,134.52822,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,RODINGA,NAMATJIRA,Not recorded,4,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/14153168,POINT (134.52822 -23.7712),70201105216,Sandover - Plenty,70900103,South MacDonnell Ranges
790,14153250,TODD RIVER DOWNS,No aliases recorded with NTG DIPL,Family Outstation,-24.25840,135.27050,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,RODINGA,NAMATJIRA,Not recorded,1,Homelands Service Provider Report 2023,https://bushtel.nt.gov.au/profile/14153250,POINT (135.2705 -24.2584),70201105216,Sandover - Plenty,70900103,South MacDonnell Ranges


In [96]:
n_no_sa1 = int(tagged["sa1_code"].isna().sum())
n_no_iloc = int(tagged["iloc_code"].isna().sum())
print(f"villages with no SA1 match: {n_no_sa1};  with no ILOC match: {n_no_iloc}")

villages with no SA1 match: 0;  with no ILOC match: 0


Exploration showed this is 0/0. Stop if a data refresh ever changes that, so we can decide what to do with the strays instead of silently losing them.

In [97]:
assert n_no_sa1 == 0 and n_no_iloc == 0, "a village fell outside the ABS polygons - investigate"

Write the village table: everything from 01, plus the two area codes.

In [98]:
village_columns = [
    "community_id", "community_name", "community_aliases", "community_type",
    "latitude", "longitude",
    "land_council", "local_govt_council", "ntg_region", "ward", "electorate", "main_language",
    "bushtel_url",
    "population_bushtel_2024", "population_bushtel_source",
    "sa1_code", "sa2_name", "iloc_code", "iloc_name",
]
village_out = (
    pd.DataFrame(tagged)[village_columns].sort_values("community_id").reset_index(drop=True)
)

In [99]:
village_out

,community_id,community_name,community_aliases,community_type,latitude,longitude,land_council,local_govt_council,ntg_region,ward,electorate,main_language,bushtel_url,population_bushtel_2024,population_bushtel_source,sa1_code,sa2_name,iloc_code,iloc_name
0,1,ADELAIDE BORE,"WOOLA,WOOLLA",Family Outstation,-22.21485,133.93607,CENTRAL LAND COUNCIL,CENTRAL DESERT,CENTRAL AUSTRALIA,ANMATJERE,BARKLY,Not recorded,https://bushtel.nt.gov.au/profile/1,14,Homelands Service Provider Report 2023,70201105410,Yuendumu - Anmatjere,70900301,Anmatjere - Surrounds
1,2,ALYUEN,"AILERON,ALUM,ALUM CAMP",Family Outstation,-22.66949,133.35147,CENTRAL LAND COUNCIL,CENTRAL DESERT,CENTRAL AUSTRALIA,ANMATJERE,BARKLY,Not recorded,https://bushtel.nt.gov.au/profile/2,12,Homelands Service Provider Report 2023,70201105410,Yuendumu - Anmatjere,70900301,Anmatjere - Surrounds
2,3,AKNGWERTNARRE,MORRIS SOAK,Town Camp,-23.69280,133.86120,CENTRAL LAND COUNCIL,ALICE SPRINGS,CENTRAL AUSTRALIA,Not recorded,BRAITLING,Not recorded,https://bushtel.nt.gov.au/profile/3,<NA>,Not recorded,70201104513,Charles,70800206,Nyewente - Akngwertnarre - Namatjira
3,4,ALKIPI,"ALKIPIE,ALKPIE,MOUNT LARRY,YALKIPI",Family Outstation,-23.27356,131.82924,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,LURITJA PINTUBI,GWOJA,Not recorded,https://bushtel.nt.gov.au/profile/4,<NA>,Not recorded,70201105310,Tanami,70901102,Papunya Outstations
4,5,RED SANDHILL,INTJIRRANGA,Family Outstation,-24.00974,132.85279,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,LJIRAPINTA,GWOJA,Not recorded,https://bushtel.nt.gov.au/profile/5,3,Homelands Service Provider Report 2023,70201105312,Tanami,70901403,Tjuwanpa Outstations
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,13339033,HONEYMOON BORE,No aliases recorded with NTG DIPL,Family Outstation,-21.66791,135.24665,CENTRAL LAND COUNCIL,BARKLY,BARKLY,ALYAWARR,BARKLY,Not recorded,https://bushtel.nt.gov.au/profile/13339033,2,Homelands Service Provider Report 2023,70201105209,Sandover - Plenty,70900201,Ampilatwatja and Outstations
788,14045564,BARNJARN,No aliases recorded with NTG DIPL,Family Outstation,-14.28264,132.08321,NORTHERN LAND COUNCIL,ROPER GULF,BIG RIVERS,NYIRRANGGULUNG,ARNHEM,Not recorded,https://bushtel.nt.gov.au/profile/14045564,8,Homelands Service Provider Report 2023,70205106503,Elsey,70500303,Elsey Roper - Surrounds
789,14153168,SALT BORE,No aliases recorded with NTG DIPL,Family Outstation,-23.77120,134.52822,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,RODINGA,NAMATJIRA,Not recorded,https://bushtel.nt.gov.au/profile/14153168,4,Homelands Service Provider Report 2023,70201105216,Sandover - Plenty,70900103,South MacDonnell Ranges
790,14153250,TODD RIVER DOWNS,No aliases recorded with NTG DIPL,Family Outstation,-24.25840,135.27050,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,RODINGA,NAMATJIRA,Not recorded,https://bushtel.nt.gov.au/profile/14153250,1,Homelands Service Provider Report 2023,70201105216,Sandover - Plenty,70900103,South MacDonnell Ranges


In [100]:
# Slim the census file down to just what we need, and rename to match villages' key
census_slim = cen21_sa1[["SA1_CODE_2021", "Tot_P_P"]].rename(
    columns={"SA1_CODE_2021": "sa1_code", "Tot_P_P": "pop_census_2021"}
)

In [101]:
village_out['sa1_code'].dtype

<StringDtype(storage='python', na_value=nan)>

In [102]:
census_slim['sa1_code'].dtype

<StringDtype(storage='python', na_value=nan)>

In [103]:
cen_merge = village_out.merge(census_slim, on='sa1_code', how='left')
# Flag column: how many villages share this SA1 (and therefore share this population figure)
cen_merge["n_villages_in_sa1"] = cen_merge.groupby("sa1_code")["community_id"].transform("count")

In [104]:
# --- Validate before trusting the result ---
assert cen_merge["pop_census_2021"].isna().sum() == 0, "some villages did not match a census SA1"
assert len(cen_merge) == len(village_out), "row count changed during merge"

In [105]:
# Keep original column order, with the two new columns appended at the end
cols = list(village_out.columns) + ["pop_census_2021", "n_villages_in_sa1"]
merged = cen_merge[cols]


In [106]:
# # Guard so this can never silently regress again.
# for c in ("sa1_code", "sa2_name", "iloc_code", "iloc_name"):
#     assert c in merged.columns, f"{c} missing from village output"

# merged.to_csv("data/new_processed/villages_with_areas.csv", index=False)

In [107]:
merged

,community_id,community_name,community_aliases,community_type,latitude,longitude,land_council,local_govt_council,ntg_region,ward,...,main_language,bushtel_url,population_bushtel_2024,population_bushtel_source,sa1_code,sa2_name,iloc_code,iloc_name,pop_census_2021,n_villages_in_sa1
0,1,ADELAIDE BORE,"WOOLA,WOOLLA",Family Outstation,-22.21485,133.93607,CENTRAL LAND COUNCIL,CENTRAL DESERT,CENTRAL AUSTRALIA,ANMATJERE,...,Not recorded,https://bushtel.nt.gov.au/profile/1,14,Homelands Service Provider Report 2023,70201105410,Yuendumu - Anmatjere,70900301,Anmatjere - Surrounds,197,16
1,2,ALYUEN,"AILERON,ALUM,ALUM CAMP",Family Outstation,-22.66949,133.35147,CENTRAL LAND COUNCIL,CENTRAL DESERT,CENTRAL AUSTRALIA,ANMATJERE,...,Not recorded,https://bushtel.nt.gov.au/profile/2,12,Homelands Service Provider Report 2023,70201105410,Yuendumu - Anmatjere,70900301,Anmatjere - Surrounds,197,16
2,3,AKNGWERTNARRE,MORRIS SOAK,Town Camp,-23.69280,133.86120,CENTRAL LAND COUNCIL,ALICE SPRINGS,CENTRAL AUSTRALIA,Not recorded,...,Not recorded,https://bushtel.nt.gov.au/profile/3,<NA>,Not recorded,70201104513,Charles,70800206,Nyewente - Akngwertnarre - Namatjira,122,2
3,4,ALKIPI,"ALKIPIE,ALKPIE,MOUNT LARRY,YALKIPI",Family Outstation,-23.27356,131.82924,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,LURITJA PINTUBI,...,Not recorded,https://bushtel.nt.gov.au/profile/4,<NA>,Not recorded,70201105310,Tanami,70901102,Papunya Outstations,65,6
4,5,RED SANDHILL,INTJIRRANGA,Family Outstation,-24.00974,132.85279,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,LJIRAPINTA,...,Not recorded,https://bushtel.nt.gov.au/profile/5,3,Homelands Service Provider Report 2023,70201105312,Tanami,70901403,Tjuwanpa Outstations,235,44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,13339033,HONEYMOON BORE,No aliases recorded with NTG DIPL,Family Outstation,-21.66791,135.24665,CENTRAL LAND COUNCIL,BARKLY,BARKLY,ALYAWARR,...,Not recorded,https://bushtel.nt.gov.au/profile/13339033,2,Homelands Service Provider Report 2023,70201105209,Sandover - Plenty,70900201,Ampilatwatja and Outstations,439,2
788,14045564,BARNJARN,No aliases recorded with NTG DIPL,Family Outstation,-14.28264,132.08321,NORTHERN LAND COUNCIL,ROPER GULF,BIG RIVERS,NYIRRANGGULUNG,...,Not recorded,https://bushtel.nt.gov.au/profile/14045564,8,Homelands Service Provider Report 2023,70205106503,Elsey,70500303,Elsey Roper - Surrounds,193,12
789,14153168,SALT BORE,No aliases recorded with NTG DIPL,Family Outstation,-23.77120,134.52822,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,RODINGA,...,Not recorded,https://bushtel.nt.gov.au/profile/14153168,4,Homelands Service Provider Report 2023,70201105216,Sandover - Plenty,70900103,South MacDonnell Ranges,364,34
790,14153250,TODD RIVER DOWNS,No aliases recorded with NTG DIPL,Family Outstation,-24.25840,135.27050,CENTRAL LAND COUNCIL,MACDONNELL,CENTRAL AUSTRALIA,RODINGA,...,Not recorded,https://bushtel.nt.gov.au/profile/14153250,1,Homelands Service Provider Report 2023,70201105216,Sandover - Plenty,70900103,South MacDonnell Ranges,364,34


### PART 2 - SA1 TABLE: census + geography + the villages inside each SA1

Start from the SA1 -> Remoteness Area correspondence workbook. It lists every NT SA1 once, with its remoteness class and its Albers area in km2.

In [108]:
ra = pd.read_excel('data/external/abs_boundaries/RA_2021_AUST.xlsx', sheet_name='SA1_RA_2021_AUST', dtype={"SA1_CODE_2021": str})

In [109]:
ra = ra[ra["STATE_NAME_2021"] == "Northern Territory"]

In [110]:
sa1_report = pd.DataFrame({
    "sa1_code": ra["SA1_CODE_2021"],
    "remoteness_name": ra["RA_NAME_2021"],
    "area_sqkm": ra["AREA_ALBERS_SQKM"].round(2),
})
print(f"{len(sa1_report)} NT SA1s from the remoteness correspondence")

649 NT SA1s from the remoteness correspondence


Add the SA2/SA3/SA4 names from the shapefile attributes (human orientation).

In [111]:
sa1_names = pd.DataFrame(sa1_polys)[["sa1_code", "sa2_name", "sa3_name", "sa4_name"]].drop_duplicates("sa1_code")
sa1_report = sa1_report.merge(sa1_names, on="sa1_code", how="left")

Median household income and the overcrowding proxy from GCP table G02.

In [112]:
g02 = pd.read_csv('data/raw/census_gcp_2021/2021_GCP_all_for_NT_short-header/2021 Census GCP All Geographies for NT/SA1/NT/2021Census_G02_NT_SA1.csv', dtype={"SA1_CODE_2021": str})
g02_slim = pd.DataFrame({
    "sa1_code": g02["SA1_CODE_2021"],
    "median_hh_income_weekly": pd.to_numeric(g02["Median_tot_hhd_inc_weekly"], errors="coerce"),
    "avg_household_size": pd.to_numeric(g02["Average_household_size"], errors="coerce"),
})
sa1_report = sa1_report.merge(g02_slim, on="sa1_code", how="left")

In [113]:
sa1_report

,sa1_code,remoteness_name,area_sqkm,sa2_name,sa3_name,sa4_name,median_hh_income_weekly,avg_household_size
0,70101100907,Outer Regional Australia,0.10,Woolner - Bayview - Winnellie,Darwin City,Darwin,2289,2.3
1,70101100906,Outer Regional Australia,4.41,Woolner - Bayview - Winnellie,Darwin City,Darwin,1016,1.6
2,70101100101,Outer Regional Australia,14.32,Darwin Airport,Darwin City,Darwin,0,0.0
3,70101100206,Outer Regional Australia,0.46,Darwin City,Darwin City,Darwin,1125,1.0
4,70102101602,Outer Regional Australia,0.13,Coconut Grove,Darwin Suburbs,Darwin,2120,2.4
...,...,...,...,...,...,...,...,...
644,70201105305,Very Remote Australia,4.74,Tanami,Alice Springs,Northern Territory - Outback,1520,4.6
645,79797979992,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),0,0.0
646,79797979991,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),0,0.0
647,79797979993,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),4150,2.3


Roll the villages up to their SA1: how many, their names, their BushTel populations.

In [114]:
pop_indig_slim = cen21_sa1[["SA1_CODE_2021", "Tot_P_P"]].rename(
    columns={"SA1_CODE_2021": "sa1_code", "Tot_P_P": "pop_census_2021"}
)

sa1_report = sa1_report.merge(
    pop_indig_slim[["sa1_code", "pop_census_2021"]],
    on="sa1_code", how="left"
)

In [115]:
sa1_report

,sa1_code,remoteness_name,area_sqkm,sa2_name,sa3_name,sa4_name,median_hh_income_weekly,avg_household_size,pop_census_2021
0,70101100907,Outer Regional Australia,0.10,Woolner - Bayview - Winnellie,Darwin City,Darwin,2289,2.3,269
1,70101100906,Outer Regional Australia,4.41,Woolner - Bayview - Winnellie,Darwin City,Darwin,1016,1.6,197
2,70101100101,Outer Regional Australia,14.32,Darwin Airport,Darwin City,Darwin,0,0.0,23
3,70101100206,Outer Regional Australia,0.46,Darwin City,Darwin City,Darwin,1125,1.0,3
4,70102101602,Outer Regional Australia,0.13,Coconut Grove,Darwin Suburbs,Darwin,2120,2.4,564
...,...,...,...,...,...,...,...,...,...
644,70201105305,Very Remote Australia,4.74,Tanami,Alice Springs,Northern Territory - Outback,1520,4.6,438
645,79797979992,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),0,0.0,14
646,79797979991,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),0,0.0,0
647,79797979993,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),4150,2.3,78


In [116]:
def join_names(names: pd.Series) -> str:
    return "; ".join(names.astype(str))

In [117]:
def join_pops(pops: pd.Series) -> str:
    # "n/a" keeps the position lined up with the village-names list
    return "; ".join("n/a" if pd.isna(p) else str(int(p)) for p in pops)

In [118]:
village_by_sa1 = tagged.groupby("sa1_code").agg(
    n_villages=("community_id", "size"),
    village_names=("community_name", join_names),
    village_bushtel_pops=("population_bushtel_2024", join_pops),
    village_bushtel_pop_sum=("population_bushtel_2024", lambda s: int(s.dropna().sum())),
    n_villages_with_bushtel_pop=("population_bushtel_2024", lambda s: int(s.notna().sum())),
)
sa1_report = sa1_report.merge(village_by_sa1, on="sa1_code", how="left")

In [119]:
sa1_report

,sa1_code,remoteness_name,area_sqkm,sa2_name,sa3_name,sa4_name,median_hh_income_weekly,avg_household_size,pop_census_2021,n_villages,village_names,village_bushtel_pops,village_bushtel_pop_sum,n_villages_with_bushtel_pop
0,70101100907,Outer Regional Australia,0.10,Woolner - Bayview - Winnellie,Darwin City,Darwin,2289,2.3,269,NaN,NaN,NaN,<NA>,<NA>
1,70101100906,Outer Regional Australia,4.41,Woolner - Bayview - Winnellie,Darwin City,Darwin,1016,1.6,197,NaN,NaN,NaN,<NA>,<NA>
2,70101100101,Outer Regional Australia,14.32,Darwin Airport,Darwin City,Darwin,0,0.0,23,NaN,NaN,NaN,<NA>,<NA>
3,70101100206,Outer Regional Australia,0.46,Darwin City,Darwin City,Darwin,1125,1.0,3,NaN,NaN,NaN,<NA>,<NA>
4,70102101602,Outer Regional Australia,0.13,Coconut Grove,Darwin Suburbs,Darwin,2120,2.4,564,NaN,NaN,NaN,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644,70201105305,Very Remote Australia,4.74,Tanami,Alice Springs,Northern Territory - Outback,1520,4.6,438,1.0,PAPUNYA,514,514,1
645,79797979992,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),0,0.0,14,NaN,NaN,NaN,<NA>,<NA>
646,79797979991,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),0,0.0,0,NaN,NaN,NaN,<NA>,<NA>
647,79797979993,Migratory - Offshore - Shipping (NT),NaN,Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),Migratory - Offshore - Shipping (NT),4150,2.3,78,NaN,NaN,NaN,<NA>,<NA>


In [120]:

# SA1s with no village: fill the counts with 0 and the text columns with "".
sa1_report["n_villages"] = sa1_report["n_villages"].fillna(0).astype(int)
sa1_report["village_bushtel_pop_sum"] = sa1_report["village_bushtel_pop_sum"].fillna(0).astype(int)
sa1_report["n_villages_with_bushtel_pop"] = sa1_report["n_villages_with_bushtel_pop"].fillna(0).astype(int)
sa1_report["village_names"] = sa1_report["village_names"].fillna("")
sa1_report["village_bushtel_pops"] = sa1_report["village_bushtel_pops"].fillna("")

Order columns, sort so the village-bearing SA1s (the ones we care about) come first, then by population. The final gap-measure sort happens in 08.

In [121]:
sa1_report = sa1_report[[
    "sa1_code", "sa2_name", "sa3_name", "sa4_name", "remoteness_name", "area_sqkm",
    "pop_census_2021", "median_hh_income_weekly", "avg_household_size",
    "n_villages", "n_villages_with_bushtel_pop", "village_bushtel_pop_sum",
    "village_names", "village_bushtel_pops",
]].sort_values(
    ["n_villages", "pop_census_2021"], ascending=[False, False]
).reset_index(drop=True)

In [122]:
sa1_polys.to_file('data/new_processed/sa1_boundaries.geojson', driver="GeoJSON")

In [ ]:
# csv_path = 'data/new_processed/sa1_report.csv'
# sa1_report.to_csv(csv_path, index=False)

# points = gpd.GeoDataFrame(
#     merged.copy(),
#     geometry=gpd.points_from_xy(merged["longitude"], merged["latitude"]),
#     crs="EPSG:4326", # coordinate reference system
# )
# geojson_path = "data/new_processed/villages_with_areas.geojson"
# points.to_file(geojson_path, driver="GeoJSON")